# TDVP (RBM) vs exact dynamics: baseline and accumulated error

Spin-1/2 chain with all-to-all XX couplings and a field along Z:

$$H = h \sum_i Z_i + \sum_{i<j} J_{ij} X_i X_j, \qquad J_{ij} = 1.$$

The initial state is a complex RBM ($\alpha = 1$) with seeded parameters (`SEED = 45`), held exactly as a `FullSumState`. Its vector is also the initial condition of the exact evolution $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$ (`scipy.sparse.linalg.expm_multiply`), so every difference comes from the TDVP projection and the integrator.

TDVP settings: RK45 with `dt = 1e-3` (forced onto the same time grid as ED), `QGTJacobianDense(holomorphic=True)` with `diag_shift = 1e-6`, SVD solver. With `diag_shift = 1e-4` the regularization alone gives an error of about $10^{-3}$ on $\langle Z \rangle$ at $t = 0.5$.

The notebook has three parts:

1. **Baseline** (N = 6, h = 10, T = 5): $\langle Z \rangle$ along the run, as in issue #2. The same comparison at $T = 0.5$ is the CI test `tests/QELM/NQS/dynamics/test_tdvp_baseline.py`.
2. **All tracked observables** for one run: energy, local magnetizations $\langle X_i \rangle$ and $\langle Z_i \rangle$, half-chain entanglement entropy, fidelity.
3. **Accumulated error against the field** $h$, for N = 4, 6 and 8.

Figures go to `outputs/tdvp_vs_ed_baseline/<timestamp>/plots/`.

In [ ]:
import numpy as np

from QELM.NQS.dynamics import (
    BASELINE_SEED,
    ERROR_KEYS,
    FieldSweep,
    TDVPSettings,
    accumulated_error_vs_field,
    compare_with_exact,
    tdvp_vs_exact,
)
from QELM.NQS.paths import experiment_paths
from QELM.NQS.utils.makeplots import plot_exact_vs_tdvp, plot_field_scan
from QELM.NQS.utils.outputs import find_repo_root

ROOT = find_repo_root()
J = 1.0
DT = 1e-3
SEED = BASELINE_SEED  # 45
SETTINGS = TDVPSettings(dt=DT, integrator="RK45", diag_shift=1e-6, linear_solver="svd")

paths = experiment_paths(
    experiment_name="tdvp_vs_ed_baseline", outputs_root=ROOT / "outputs"
)
print("run folder:", paths.root)

## 1. Baseline: N = 6, h = 10, T = 5

In [ ]:
result = tdvp_vs_exact(
    n_spins=6, h=10.0, J=J, T=5.0, seed=SEED, settings=SETTINGS, paths=paths
)
print("time points (ED, TDVP):", len(result.times), len(result.tdvp_times))
print(f"max |<Z>_TDVP - <Z>_ED| = {result.max_abs_error:.2e}")
print(f"final fidelity |<psi_ED(T)|psi_TDVP(T)>|^2 = {result.final_fidelity:.6f}")
drift = abs(result.tdvp_energy - result.exact_energy[0]).max()
print(f"energy drift max |E_TDVP(t) - E(0)| = {drift:.2e}")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(result.plot)))

## 2. All tracked observables for one run

`compare_with_exact` records the TDVP state vector on the grid (`run_tdvp(..., record_states=True)`) and evaluates every observable on both trajectories with the same code:

- the energy $\langle H \rangle$ (conserved by both dynamics, up to the regularization for TDVP);
- the local magnetizations $\langle X_i \rangle$ and $\langle Z_i \rangle$;
- the half-chain von Neumann entropy $S = -\mathrm{Tr}\,\rho_A \ln \rho_A$ of sites $0, \dots, N/2 - 1$, in nats (at most $\frac{N}{2}\ln 2$);
- the fidelity $F(t) = |\langle \psi_\mathrm{ED}(t) | \psi_\mathrm{TDVP}(t) \rangle|^2$.

Here N = 6, h = 1, T = 2: at $h = J$ the field and the couplings compete.

In [ ]:
example = compare_with_exact(n_spins=6, h=1.0, J=J, T=2.0, seed=SEED, settings=SETTINGS)
t = example.times
panels = {
    "energy": (example.exact.energy, example.tdvp.energy, r"$\langle H \rangle$"),
    "sigma_x0": (
        example.exact.sigma_x[:, 0],
        example.tdvp.sigma_x[:, 0],
        r"$\langle X_0 \rangle$",
    ),
    "sigma_z0": (
        example.exact.sigma_z[:, 0],
        example.tdvp.sigma_z[:, 0],
        r"$\langle Z_0 \rangle$",
    ),
    "entropy": (example.exact.entropy, example.tdvp.entropy, r"$S_{N/2}$ (nats)"),
}
for name, (exact, tdvp, label) in panels.items():
    path = plot_exact_vs_tdvp(
        t,
        exact,
        t,
        tdvp,
        filename=f"example_{name}.png",
        directory=paths.plots,
        ylabel=label,
        title=f"TDVP vs ED (N=6, h=1): {label}",
    )
    display(Image(filename=str(path)))

print(f"min fidelity over the run: {example.fidelity.min():.6f}")
for key, value in example.accumulated_errors().items():
    print(
        f"accumulated error {key:>16}: {value:.3e}   (time average {value / t[-1]:.3e})"
    )

## 3. Accumulated absolute error against the field

For each observable $O$, the accumulated absolute error over the run is

$$\mathcal{E}_O(T) = \int_0^T \big|\langle O \rangle_\mathrm{TDVP}(t) - \langle O \rangle_\mathrm{ED}(t)\big|\, dt$$

(trapezoidal rule on the `dt = 1e-3` grid, units of $[O] \times$ time; divide by $T$ for the time average). For the local magnetizations the error is averaged over sites, $\frac{1}{N} \sum_i \mathcal{E}_{X_i}$. The error of the mean magnetization $\frac{1}{N}\sum_i Z_i$ is at most the site-averaged one (triangle inequality). The infidelity is accumulated as $\int_0^T (1 - F)\, dt$.

Every run starts from the same seeded RBM; only $h$ changes. A sweep takes about 10 to 25 minutes per N on a laptop CPU, so each one is cached in `outputs/tdvp_vs_ed_error_scan/` (gitignored). Set `RECOMPUTE = True` to run them again.

In [ ]:
SIZES = (4, 6, 8)
FIELDS = (0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0)
T_SCAN = 2.0
RECOMPUTE = False

cache = ROOT / "outputs" / "tdvp_vs_ed_error_scan"
sweeps: dict[int, FieldSweep] = {}
for n in SIZES:
    file = cache / f"sweep_N{n}_T{T_SCAN:g}_seed{SEED}.npz"
    if file.is_file() and not RECOMPUTE:
        sweep = FieldSweep.load(file)
        assert np.allclose(sweep.fields, FIELDS) and sweep.T == T_SCAN
    else:
        sweep = accumulated_error_vs_field(
            n_spins=n, fields=FIELDS, J=J, T=T_SCAN, seed=SEED, settings=SETTINGS
        )
        sweep.save(file)
    sweeps[n] = sweep
    print(f"N={n}: total TDVP time {sweep.wall_time.sum():.0f} s")

In [ ]:
labels = {
    "energy": r"$\int |\Delta \langle H \rangle|\, dt$",
    "magnetization_z": r"$\int |\Delta \langle \bar Z \rangle|\, dt$",
    "sigma_x": r"$\frac{1}{N}\sum_i \int |\Delta \langle X_i \rangle|\, dt$",
    "sigma_z": r"$\frac{1}{N}\sum_i \int |\Delta \langle Z_i \rangle|\, dt$",
    "entropy": r"$\int |\Delta S_{N/2}|\, dt$",
    "infidelity": r"$\int (1 - F)\, dt$",
}
scan = plot_field_scan(
    {f"N={n}": s.fields for n, s in sweeps.items()},
    {f"N={n}": s.errors for n, s in sweeps.items()},
    panels=labels,
    filename="accumulated_error_vs_h.png",
    directory=paths.plots,
    title=f"Accumulated TDVP-vs-ED error, T={T_SCAN:g}, dt={DT:g}, diag_shift=1e-6",
)
display(Image(filename=str(scan)))

In [ ]:
header = (
    f"{'N':>2} {'h':>5} " + " ".join(f"{k:>15}" for k in ERROR_KEYS) + f" {'F(T)':>10}"
)
print(header)
for n, s in sweeps.items():
    for i, h in enumerate(s.fields):
        row = " ".join(f"{s.errors[k][i]:15.3e}" for k in ERROR_KEYS)
        print(f"{n:>2} {h:5.1f} {row} {s.final_fidelity[i]:10.6f}")

### Reading the scan

INTERPRETATION_PLACEHOLDER